Sentimen Analisis Review Aplikasi Discord (Google Play Store)

# Install Module

In [123]:
!pip install seaborn
!pip install sastrawi
!pip install wordcloud
!pip install gensim


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Import Module

In [124]:
import datetime as dt  # Manipulasi data waktu dan tanggal
import re  # Modul untuk bekerja dengan ekspresi reguler
import string  # Berisi konstanta string, seperti tanda baca
from nltk.tokenize import word_tokenize  # Tokenisasi teks
from nltk.corpus import stopwords  # Daftar kata-kata berhenti dalam teks
 
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory  # Stemming (penghilangan imbuhan kata) dalam bahasa Indonesia
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory  # Menghapus kata-kata berhenti dalam bahasa Indonesia
 
from wordcloud import WordCloud  # Membuat visualisasi berbentuk awan kata (word cloud) dari teks

import pandas as pd  # Pandas untuk manipulasi dan analisis data
pd.options.mode.chained_assignment = None  # Menonaktifkan peringatan chaining
import numpy as np  # NumPy untuk komputasi numerik
seed = 0
np.random.seed(seed)  # Mengatur seed untuk reproduktibilitas
from sklearn.metrics import accuracy_score

import nltk  # Import pustaka NLTK
nltk.download('punkt_tab')  # Mengunduh dataset yang diperlukan untuk tokenisasi teks.
nltk.download('stopwords')  # Mengunduh dataset yang berisi daftar kata-kata berhenti (stopwords) dalam berbagai bahasa.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kenny\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kenny\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Preprocessing Data

In [125]:
# Memuat dataset
app_reviews_df = pd.read_csv('D:/Latihan/Deep-Learning/proyek_1/ulasan_aplikasi_discord.csv')

C:\Users\kenny\AppData\Local\Temp\ipykernel_3628\58789662.py:2: DtypeWarning: Columns (0: reviewCreatedVersion, 1: appVersion) have mixed types. Specify dtype option on import or set low_memory=False.
  app_reviews_df = pd.read_csv('D:/Latihan/Deep-Learning/proyek_1/ulasan_aplikasi_discord.csv')


In [126]:
# hapus data NAN
clean_df = app_reviews_df.dropna()

In [127]:
# Hapus baris duplikat dari DataFrame clean_df
clean_df = clean_df.drop_duplicates()
 
# Hitung jumlah baris dan kolom dalam DataFrame clean_df setelah menghapus duplikat
jumlah_ulasan_setelah_hapus_duplikat, jumlah_kolom_setelah_hapus_duplikat = clean_df.shape

In [128]:
clean_df.info()

<class 'pandas.DataFrame'>
Index: 11456 entries, 0 to 60020
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   reviewId              11456 non-null  str  
 1   userName              11456 non-null  str  
 2   userImage             11456 non-null  str  
 3   content               11456 non-null  str  
 4   score                 11456 non-null  int64
 5   thumbsUpCount         11456 non-null  int64
 6   reviewCreatedVersion  11456 non-null  str  
 7   at                    11456 non-null  str  
 8   replyContent          11456 non-null  str  
 9   repliedAt             11456 non-null  str  
 10  appVersion            11456 non-null  str  
dtypes: int64(2), str(9)
memory usage: 6.9 MB


In [129]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
 
def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # menghapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # menghapus hashtag
    text = re.sub(r'RT[\s]', '', text) # menghapus RT
    text = re.sub(r"http\S+", '', text) # menghapus link
    text = re.sub(r'[0-9]+', '', text) # menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka
 
    text = text.replace('\n', ' ') # mengganti baris baru dengan spasi
    text = text.translate(str.maketrans('', '', string.punctuation)) # menghapus semua tanda baca
    text = text.strip(' ') # menghapus karakter spasi dari kiri dan kanan teks
    return text
 
def casefoldingText(text): # Mengubah semua karakter dalam teks menjadi huruf kecil
    text = text.lower()
    return text
 
def tokenizingText(text): # Memecah atau membagi string, teks menjadi daftar token
    text = word_tokenize(text)
    return text
 
def filteringText(text): # Menghapus stopwords dalam teks
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords1 = set(stopwords.words('english'))
    listStopwords.update(listStopwords1)
    listStopwords.update(["aja", "aku", "anda", "atau", "banget", "baru", "bisa",
    "buat", "dan", "dari", "dengan", "di", "dulu", "ga",
    "gak", "ini", "itu", "jadi", "juga", "kalau", "kalo",
    "kami", "karena", "ke", "kita", "kok", "lagi", "lah",
    "masih", "mau", "nya", "padahal", "pas", "sama", "saya",
    "sih", "sudah", "terus",  "udah", "untuk",
    "ya", "yang"
])

    filtered = []
    for txt in text:
        if txt not in listStopwords:
            filtered.append(txt)
    text = filtered
    return text
 
def stemmingText(text): # Mengurangi kata ke bentuk dasarnya yang menghilangkan imbuhan awalan dan akhiran atau ke akar kata
    # Membuat objek stemmer
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()
 
    # Memecah teks menjadi daftar kata
    words = text.split()
 
    # Menerapkan stemming pada setiap kata dalam daftar
    stemmed_words = [stemmer.stem(word) for word in words]
 
    # Menggabungkan kata-kata yang telah distem
    stemmed_text = ' '.join(stemmed_words)
 
    return stemmed_text
 
def toSentence(list_words): # Mengubah daftar kata menjadi kalimat
    sentence = ' '.join(word for word in list_words)
    return sentence

In [130]:
slangwords = {
    "ga" : "tidak",
    "gak" : "tidak",
    "gk" : "tidak",
    "udah" : "sudah", 
    "udh" : "sudah",
    "gw" : "saya",
    "gua" : "saya",
    "loe" : "kamu",
    "lu" : "kamu",
    "kalo" : "kalau",
    "tpi" : "tapi",
    "tp" : "tapi",
    "yg" : "yang",
    "jg" : "juga",
    "bgt" : "banget",
    "sihh" : "sih",
    "mulu" : "terus",
    "temen" : "teman",
    "pls" : "please",
    "pdhl" : "padahal",
    "gt" : "gitu",
    "gtu" : "gitu",
    "trs" : "terus",
    "lgi" : "lagi",
    "gabisa" : "tidak bisa",
    "kagak" : "tidak",
    "kaga" : "tidak",
    "dah" : "sudah",
    "knp" : "kenapa",
    "gimana" : "bagaimana",
    "kasi" : "kasih",
    "pake" : "pakai",
    "ny" : "nya",
    "jir" : "jir",
    "cuman" : "cuma",
    "sampe" : "sampai",
    "dri" : "dari",
    "skrg" : "sekarang",
    "dmn" : "di mana",
    "sma" : "sama",
    "gda" : "tidak ada",
    "krna" : "karena",
    "nggak" : "tidak",
    "ngirim" : "mengirim",
    "ngelag" : "mengelag",
    "ngobrol" : "mengobrol",
    "ngambil" : "mengambil",
    "nyoba" : "mencoba",
    "benerin" : "membenarkan",
    "tolongin" : "tolong"
}

In [131]:
def fixSlangwords(text):
    words = text.split()
    fixed_words = []
 
    for word in words:
        if word.lower() in slangwords:
            fixed_words.append(slangwords[word.lower()])
        else:
            fixed_words.append(word)
 
    fixed_text = ' '.join(fixed_words)
    return fixed_text

In [132]:
# Membersihkan teks dan menyimpannya di kolom 'text_clean'
clean_df['text_clean'] = clean_df['content'].apply(cleaningText)
 
# Mengubah huruf dalam teks menjadi huruf kecil dan menyimpannya di 'text_casefoldingText'
clean_df['text_casefoldingText'] = clean_df['text_clean'].apply(casefoldingText)
 
# Mengganti kata-kata slang dengan kata-kata standar dan menyimpannya di 'fixSlangwords'
clean_df['text_slangwords'] = clean_df['text_casefoldingText'].apply(fixSlangwords)
 
# Memecah teks menjadi token (kata-kata) dan menyimpannya di 'text_tokenizingText'
clean_df['text_tokenizingText'] = clean_df['text_slangwords'].apply(tokenizingText)
 
# Menghapus kata-kata stop (kata-kata umum) dan menyimpannya di 'text_stopword'
clean_df['text_stopword'] = clean_df['text_tokenizingText'].apply(filteringText)
 
# Menggabungkan token-token menjadi kalimat dan menyimpannya di 'text_final'
clean_df['text_final'] = clean_df['text_stopword'].apply(toSentence)

In [133]:
# Cek dataframe clean_df setelah proses pembersihan teks
print(clean_df.info())

<class 'pandas.DataFrame'>
Index: 11456 entries, 0 to 60020
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              11456 non-null  str   
 1   userName              11456 non-null  str   
 2   userImage             11456 non-null  str   
 3   content               11456 non-null  str   
 4   score                 11456 non-null  int64 
 5   thumbsUpCount         11456 non-null  int64 
 6   reviewCreatedVersion  11456 non-null  str   
 7   at                    11456 non-null  str   
 8   replyContent          11456 non-null  str   
 9   repliedAt             11456 non-null  str   
 10  appVersion            11456 non-null  str   
 11  text_clean            11456 non-null  str   
 12  text_casefoldingText  11456 non-null  str   
 13  text_slangwords       11456 non-null  str   
 14  text_tokenizingText   11456 non-null  object
 15  text_stopword         11456 non-null  object
 16  te

# Labeling

In [134]:
import csv
import requests
from io import StringIO
 
# Membaca data kamus kata-kata positif dari GitHub
lexicon_positive = dict()
 
response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv')
# Mengirim permintaan HTTP untuk mendapatkan file CSV dari GitHub
 
if response.status_code == 200:
    # Jika permintaan berhasil
    reader = csv.reader(StringIO(response.text), delimiter=',')
    # Membaca teks respons sebagai file CSV menggunakan pembaca CSV dengan pemisah koma
 
    for row in reader:
        # Mengulangi setiap baris dalam file CSV
        lexicon_positive[row[0]] = int(row[1])
        # Menambahkan kata-kata positif dan skornya ke dalam kamus lexicon_positive
else:
    print("Failed to fetch positive lexicon data")
 
# Membaca data kamus kata-kata negatif dari GitHub
lexicon_negative = dict()
 
response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv')
# Mengirim permintaan HTTP untuk mendapatkan file CSV dari GitHub
 
if response.status_code == 200:
    # Jika permintaan berhasil
    reader = csv.reader(StringIO(response.text), delimiter=',')
    # Membaca teks respons sebagai file CSV menggunakan pembaca CSV dengan pemisah koma
 
    for row in reader:
        # Mengulangi setiap baris dalam file CSV
        lexicon_negative[row[0]] = int(row[1])
        # Menambahkan kata-kata negatif dan skornya dalam kamus lexicon_negative
else:
    print("Failed to fetch negative lexicon data")

In [135]:
# Fungsi untuk menentukan polaritas sentimen dari tweet
 
def sentiment_analysis_lexicon_indonesia(text):
    #for word in text:
 
    score = 0
    # Inisialisasi skor sentimen ke 0
 
    for word in text:
        # Mengulangi setiap kata dalam teks
 
        if (word in lexicon_positive):
            score = score + lexicon_positive[word]
            # Jika kata ada dalam kamus positif, tambahkan skornya ke skor sentimen
 
    for word in text:
        # Mengulangi setiap kata dalam teks (sekali lagi)
 
        if (word in lexicon_negative):
            score = score + lexicon_negative[word]
            # Jika kata ada dalam kamus negatif, kurangkan skornya dari skor sentimen
 
    polarity=''
    # Inisialisasi variabel polaritas
 
    if (score > 0):
        polarity = 'positive'
        # Jika skor sentimen lebih besar atau sama dengan 0, maka polaritas adalah positif
    elif (score < 0):
        polarity = 'negative'
        # Jika skor sentimen kurang dari 0, maka polaritas adalah negatif
    else:
        polarity = 'neutral'
    # Jika skor sentimen sama dengan 0, maka polaritas adalah netral
 
    return score, polarity
    # Mengembalikan skor sentimen dan polaritas teks

In [136]:
results = clean_df['text_stopword'].apply(sentiment_analysis_lexicon_indonesia)
results = list(zip(*results))
clean_df['polarity_score'] = results[0]
clean_df['polarity'] = results[1]

In [137]:
# Menampilkan data di dalam DataFrame app_reviews_df
clean_df

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,text_clean,text_casefoldingText,text_slangwords,text_tokenizingText,text_stopword,text_final,polarity_score,polarity
0,cf0348fe-ceca-4a51-8bcd-fa560d7748f6,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Di handphone saya lagi error temen temen lgi v...,3,5,332.12 - Stable,2026-06-16 14:44:51,"Coba restart HP, update OS, bersihkan cache/da...",2026-06-16 15:15:33,332.12 - Stable,Di handphone saya lagi error temen temen lgi v...,di handphone saya lagi error temen temen lgi v...,di handphone saya lagi error teman teman lagi ...,"[di, handphone, saya, lagi, error, teman, tema...","[handphone, error, teman, teman, voice, voice,...",handphone error teman teman voice voice hapus ...,-13,negative
3,b14c042c-1128-4b1d-9f81-d1908100adb3,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Setelah beralih ke aplikasi lain saat berbagi ...,1,1,336.10 - Stable,2026-07-08 14:45:50,"Coba update OS dan aplikasi Discord, cek izin ...",2026-07-08 15:13:29,336.10 - Stable,Setelah beralih ke aplikasi lain saat berbagi ...,setelah beralih ke aplikasi lain saat berbagi ...,setelah beralih ke aplikasi lain saat berbagi ...,"[setelah, beralih, ke, aplikasi, lain, saat, b...","[beralih, aplikasi, berbagi, layar, discord, k...",beralih aplikasi berbagi layar discord kehilan...,-9,negative
5,af6626c4-990c-4c97-81bb-5ca3398da60c,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"agak gimana ya.. bagus sih, tpi pas aku shares...",1,8,332.11 - Stable,2026-06-11 01:59:13,"Untuk masalah suara saat sharescreen, coba uni...",2026-06-11 02:23:37,332.11 - Stable,agak gimana ya bagus sih tpi pas aku sharescre...,agak gimana ya bagus sih tpi pas aku sharescre...,agak bagaimana ya bagus sih tapi pas aku share...,"[agak, bagaimana, ya, bagus, sih, tapi, pas, a...","[bagus, sharescreen, android, mobile, apk, dc,...",bagus sharescreen android mobile apk dc suaran...,-6,negative
6,ee7c8f59-fe47-4b13-991e-ed227c29ad97,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,dibagian sharescreen tidak ada suara diaplikas...,2,4,334.11 - Stable,2026-06-27 01:38:05,Untuk masalah suara saat screenshare di aplika...,2026-06-27 02:09:31,334.11 - Stable,dibagian sharescreen tidak ada suara diaplikas...,dibagian sharescreen tidak ada suara diaplikas...,dibagian sharescreen tidak ada suara diaplikas...,"[dibagian, sharescreen, tidak, ada, suara, dia...","[dibagian, sharescreen, suara, diaplikasi, you...",dibagian sharescreen suara diaplikasi youtube ...,-5,negative
7,a4eb20d5-6915-4bc4-9a9d-c48e7b3e3f84,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"suka bug. keluar sendiri, tampilan lama panggi...",2,17,325.10 - Stable,2026-04-26 22:41:57,"Coba restart aplikasi, update ke versi terbaru...",2026-04-26 23:09:28,325.10 - Stable,suka bug keluar sendiri tampilan lama panggila...,suka bug keluar sendiri tampilan lama panggila...,suka bug keluar sendiri tampilan lama panggila...,"[suka, bug, keluar, sendiri, tampilan, lama, p...","[suka, bug, tampilan, panggilan, suka, hilang,...",suka bug tampilan panggilan suka hilang teman ...,6,positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60003,1f079900-c9be-4628-95da-0af62a9a2e47,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Need friend ^^ #7133,4,1,2.7.8,2016-07-16 21:41:44,"Hey Naufal,\n\nDon't forget to put your userna...",2016-07-18 05:45:05,2.7.8,Need friend,need friend,need friend,"[need, friend]","[need, friend]",need friend,0,neutral
60013,82d2d9d0-8a2a-45ae-a3f2-621438e9d449,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,suara ny kurang jelas,2,2,2.4.4,2016-04-21 21:56:14,"Hey Annisyah,\n\nSorry for the delay in respon...",2016-06-16 06:34:51,2.4.4,suara ny kurang jelas,suara ny kurang jelas,suara nya kurang jelas,"[suara, nya, kurang, jelas]",[suara],suara,1,positive

In [138]:
print(clean_df['polarity'].value_counts())
print(clean_df['polarity'].value_counts(normalize=True))

polarity
negative    6324
neutral     2963
positive    2169
Name: count, dtype: int64
polarity
negative    0.552025
neutral     0.258642
positive    0.189333
Name: proportion, dtype: float64


# Exctraction

In [139]:
# Ekstraksi fitur dengan Word2Vec
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk
import numpy as np

nltk.download('punkt_tab')

tokenized_data = [word_tokenize(text) for text in clean_df['text_final']]

w2v = Word2Vec(sentences=tokenized_data, vector_size=100, window=5, min_count=2, workers=4, seed=42)

def doc_vector_avg(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

# Bangun matriks fitur Word2Vec untuk setiap dokumen
X_w2v = np.vstack([doc_vector_avg(toks, w2v) for toks in clean_df['text_final']])

# Konversi matriks Word2Vec menjadi DataFrame dengan kolom berlabel
w2v_cols = [f'w2v_{i}' for i in range(w2v.vector_size)]
w2v_df = pd.DataFrame(X_w2v, columns=w2v_cols)

# Menampilkan beberapa baris dari DataFrame Word2Vec
w2v_df.head()

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kenny\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,w2v_0,w2v_1,w2v_2,w2v_3,w2v_4,w2v_5,w2v_6,w2v_7,w2v_8,w2v_9,...,w2v_90,w2v_91,w2v_92,w2v_93,w2v_94,w2v_95,w2v_96,w2v_97,w2v_98,w2v_99
0,0.073988,0.014184,-0.084202,-0.046877,-0.264821,0.036284,0.085389,0.186136,-0.124235,0.057938,...,-0.211312,0.270866,0.055818,-0.141236,0.112413,-0.060380,-0.004823,0.152413,-0.035970,-0.027342
1,0.070671,0.017778,-0.086421,-0.041974,-0.266700,0.034737,0.083027,0.189952,-0.123379,0.055431,...,-0.223359,0.275553,0.059983,-0.138682,0.113190,-0.057907,-0.000332,0.153536,-0.034080,-0.026526
2,0.077302,0.015462,-0.089633,-0.048532,-0.279796,0.037512,0.090571,0.197721,-0.131552,0.061419,...,-0.225499,0.286369,0.060157,-0.148480,0.118546,-0.064091,-0.002250,0.162886,-0.037913,-0.028242
3,0.075861,0.014354,-0.086277,-0.046429,-0.270551,0.036730,0.086281,0.190815,-0.126747,0.059232,...,-0.218099,0.278589,0.058693,-0.143683,0.115044,-0.061397,-0.004009,0.155327,-0.036407,-0.028079
4,0.090041,0.018799,-0.105495,-0.052658,-0.330808,0.044486,0.102718,0.232973,-0.157433,0.069415,...,-0.271476,0.340720,0.070356,-0.173689,0.139792,-0.075684,-0.002159,0.189222,-0.044387,-0.033858


In [140]:
X = clean_df['text_final']
y = clean_df['polarity']

In [141]:
# Ekstraksi fitur dengan TF-IDF
tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95, max_features=3000)
X_tfidf = tfidf.fit_transform(X)
 
# Konversi hasil ekstraksi fitur menjadi dataframe
features_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())
 
# Menampilkan hasil ekstraksi fitur
features_df.head()

,abis,abis update,able,acc,accept,account,account already,account cant,account disabled,account discord,...,yak,yatim,yaudah,years,yes,yesterday,youre,youtube,yt,zoom
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.268057,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0


# Build & Evaluation Model (Word2Vec Features)

In [142]:
# Bagi data menjadi data latih dan data uji (Word2Vec Features) 
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_w2v, y, test_size=0.3, random_state=42, stratify=y)

In [143]:
# Import model Random Forest
from sklearn.ensemble import RandomForestClassifier

# Membuat objek model Random Forest for Word2Vec Features
random_forest_w2v = RandomForestClassifier()

# Melatih model Random Forest pada data pelatihan
random_forest_w2v.fit(X_train2, y_train2)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_rf_w2v = random_forest_w2v.predict(X_train2)
y_pred_test_rf_w2v = random_forest_w2v.predict(X_test2)

# Evaluasi akurasi model Random Forest
accuracy_train_rf_w2v = accuracy_score(y_pred_train_rf_w2v, y_train2)
accuracy_test_rf_w2v = accuracy_score(y_pred_test_rf_w2v, y_test2)

# Menampilkan akurasi
print('Random Forest x Word2Vec - accuracy_train:', accuracy_train_rf_w2v)
print('Random Forest x Word2Vec - accuracy_test:', accuracy_test_rf_w2v)

Random Forest x Word2Vec - accuracy_train: 0.9826661678513531
Random Forest x Word2Vec - accuracy_test: 0.6162350887401804


# Build & Evaluation Model (TF-IDF Features)

In [144]:
# Bagi data menjadi data latih dan data uji (TF-IDF Features)) 
X_train1, X_test1, y_train1, y_test1 = train_test_split(X_tfidf, y, test_size=0.2, random_state=42, stratify=y)

In [145]:
# Import model Naive Bayes
from sklearn.naive_bayes import MultinomialNB
 
# Membuat objek model Naive Bayes 
naive_bayes = MultinomialNB()
 
# Melatih model Naive Bayes pada data pelatihan
naive_bayes.fit(X_train1.toarray(), y_train1)
 
# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_nb = naive_bayes.predict(X_train1.toarray())
y_pred_test_nb = naive_bayes.predict(X_test1.toarray())
 
# Evaluasi akurasi model Naive Bayes
accuracy_train_nb = accuracy_score(y_pred_train_nb, y_train1)
accuracy_test_nb = accuracy_score(y_pred_test_nb, y_test1)
 
# Menampilkan akurasi
print('Naive Bayes - accuracy_train:', accuracy_train_nb)
print('Naive Bayes - accuracy_test:', accuracy_test_nb)

Naive Bayes - accuracy_train: 0.755674378000873
Naive Bayes - accuracy_test: 0.6958987783595113


In [146]:
# Import model Logistic Regression
from sklearn.linear_model import LogisticRegression
 
# Membuat objek model Logistic Regression
logistic_regression = LogisticRegression()
 
# Melatih model Logistic Regression pada data pelatihan
logistic_regression.fit(X_train1.toarray(), y_train1)
 
# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_lr = logistic_regression.predict(X_train1.toarray())
y_pred_test_lr = logistic_regression.predict(X_test1.toarray())
 
# Evaluasi akurasi model Logistic Regression 
accuracy_train_lr = accuracy_score(y_pred_train_lr, y_train1)
accuracy_test_lr = accuracy_score(y_pred_test_lr, y_test1)
 
# Menampilkan akurasi
print('Logistic Regression - accuracy_train:', accuracy_train_lr)
print('Logistic Regression - accuracy_test:', accuracy_test_lr)

Logistic Regression - accuracy_train: 0.9202313400261894
Logistic Regression - accuracy_test: 0.8547120418848168


In [147]:
# Import model Decision Tree
from sklearn.tree import DecisionTreeClassifier
 
# Membuat objek model Decision Tree
decision_tree = DecisionTreeClassifier()
 
# Melatih model Decision Tree pada data pelatihan
decision_tree.fit(X_train1.toarray(), y_train1)
 
# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_dt = decision_tree.predict(X_train1.toarray())
y_pred_test_dt = decision_tree.predict(X_test1.toarray())
 
# Evaluasi akurasi model Decision Tree
accuracy_train_dt = accuracy_score(y_pred_train_dt, y_train1)
accuracy_test_dt = accuracy_score(y_pred_test_dt, y_test1)
 
# Menampilkan akurasi
print('Decision Tree - accuracy_train:', accuracy_train_dt)
print('Decision Tree - accuracy_test:', accuracy_test_dt)

Decision Tree - accuracy_train: 0.9937800087298123
Decision Tree - accuracy_test: 0.8263525305410122


In [148]:
# Import model Random Forest
from sklearn.ensemble import RandomForestClassifier
 
# Membuat objek model Random Forest
random_forest = RandomForestClassifier()
 
# Melatih model Random Forest pada data pelatihan
random_forest.fit(X_train1.toarray(), y_train1)
 
# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_rf = random_forest.predict(X_train1.toarray())
y_pred_test_rf = random_forest.predict(X_test1.toarray())
 
# Evaluasi akurasi model Random Forest
accuracy_train_rf = accuracy_score(y_pred_train_rf, y_train1)
accuracy_test_rf = accuracy_score(y_pred_test_rf, y_test1)
 
# Menampilkan akurasi
print('Random Forest - accuracy_train:', accuracy_train_rf)
print('Random Forest - accuracy_test:', accuracy_test_rf)

Random Forest - accuracy_train: 0.9937800087298123
Random Forest - accuracy_test: 0.8534031413612565


# Report Evalution

In [149]:
# Import library untuk membuat classification report
from sklearn.metrics import classification_report

# Random Forest with Word2Vec
print('Classification Report for Random Forest (Word2Vec Features):')
print(classification_report(y_test2, y_pred_test_rf_w2v))

# Random Forest with TF-IDF
print('Classification Report for Random Forest (TF-IDF Features):')
print(classification_report(y_test1, y_pred_test_rf))

# Naive Bayes with TF-IDF
print('\nClassification Report for Naive Bayes (TF-IDF Features):')
print(classification_report(y_test1, y_pred_test_nb))

# Logistic Regression with TF-IDF
print('\nClassification Report for Logistic Regression (TF-IDF Features):')
print(classification_report(y_test1, y_pred_test_lr))

# Decision Tree with TF-IDF
print('\nClassification Report for Decision Tree (TF-IDF Features):')
print(classification_report(y_test1, y_pred_test_dt))

Classification Report for Random Forest (Word2Vec Features):
              precision    recall  f1-score   support

    negative       0.62      0.89      0.73      1897
     neutral       0.63      0.41      0.50       889
    positive       0.41      0.09      0.15       651

    accuracy                           0.62      3437
   macro avg       0.55      0.46      0.46      3437
weighted avg       0.59      0.62      0.56      3437

Classification Report for Random Forest (TF-IDF Features):
              precision    recall  f1-score   support

    negative       0.87      0.91      0.89      1265
     neutral       0.82      0.88      0.85       593
    positive       0.85      0.65      0.73       434

    accuracy                           0.85      2292
   macro avg       0.85      0.81      0.82      2292
weighted avg       0.85      0.85      0.85      2292


Classification Report for Naive Bayes (TF-IDF Features):
              precision    recall  f1-score   support

    n